# NB06 — Faithfulness Aggregation and Statistical Analysis

This notebook includes the missing mIoU generation step, fixes the deletion metric schema, and makes the classification schema robust.

## Cell 1 — Mount Drive

Attach Drive and load shared project config.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())

Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


## Cell 2 — Imports and paths

Load the libraries used for aggregation, bounding-box mIoU, and statistical testing.

In [ ]:
import warnings
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr
from statsmodels.stats.multitest import multipletests


ROOT = Path(GDRIVE_ROOT)
RESULTS_PATH = ROOT / 'results'
DATA_PATH = ROOT / 'data' / 'processed'
IG_PATH = ROOT / 'ig_maps'
ANN_PATH = ROOT / 'data' / 'processed' / 'consensus'

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

ALPHA       = 0.05
BOOTSTRAP_N = 1000
MIN_TEST_N  = 30          # your individual reporting threshold
WILCOX_MIN_N = 15         # minimum paired images required for Wilcoxon
SEED        = 42
MODEL_ORDER = ['densenet121', 'convnextv2_tiny', 'swinb_lora']

print('✅ Imports and paths ready.')

✅ Imports and paths ready.


## Cell 3 — Build mIoU file

This cell computes `miou_results.csv` from NB04 IG masks and NB01 radiologist bounding boxes.

In [ ]:
def load_bbox_table(consensus='2of3'):
    path = ANN_PATH / f'consensus_boxes_{consensus}.csv'
    assert path.exists(), f'Missing: {path}'
    return pd.read_csv(path), path

def find_box_columns(df):
    colmap = {}
    for target, aliases in {'image_id':['image_id','id'], 'target_class':['target_class','class','pathology','class_name'], 'xmin':['xmin','x1','left', 'x_min'], 'ymin':['ymin','y1','top', 'y_min'], 'xmax':['xmax','x2','right', 'x_max'], 'ymax':['ymax','y2','bottom', 'y_max']}.items():
        found = next((c for c in aliases if c in df.columns), None)
        if found is None:
            raise ValueError(f'Missing bbox column for {target}. Found {df.columns.tolist()}')
        colmap[target] = found
    return colmap

def compute_miou_file(consensus='2of3'):
    bbox_df, bbox_src = load_bbox_table(consensus)
    cols = find_box_columns(bbox_df)
    manifest_path = RESULTS_PATH / 'ig_manifest.csv'
    if not manifest_path.exists():
        raise FileNotFoundError('ig_manifest.csv is required before building mIoU.')
    manifest = pd.read_csv(manifest_path)
    MASK_RES = 224
    rows = []
    for r in manifest.itertuples(index=False):
        if r.subset != 'patho':
            continue
        gt = bbox_df[(bbox_df[cols['image_id']] == r.image_id) & (bbox_df[cols['target_class']] == r.target_class)]
        if gt.empty:
            continue
        mask_path = Path(r.top50_path)
        if not mask_path.exists():
            continue

        pred = np.load(mask_path)

        # NEW: Compute exact pixel-level IoU instead of fragile bounding boxes
        # Option A: Use top50_path directly without arbitrary percentile
        # threshold = np.percentile(pred, 90)
        pred_mask = pred.astype(np.uint8)  # top50_path is already a binary mask of top 50% mass

        # Rasterize all GT boxes onto a single blank canvas
        gt_mask = np.zeros((MASK_RES, MASK_RES), dtype=np.uint8)
        for _, g in gt.iterrows():
            x1 = int(max(0, float(g[cols['xmin']])))
            y1 = int(max(0, float(g[cols['ymin']])))
            x2 = int(min(MASK_RES, float(g[cols['xmax']])))
            y2 = int(min(MASK_RES, float(g[cols['ymax']])))
            if x1 < x2 and y1 < y2:
                gt_mask[y1:y2, x1:x2] = 1

        inter = np.logical_and(pred_mask, gt_mask).sum()
        uni = np.logical_or(pred_mask, gt_mask).sum()
        miou = float(inter / uni) if uni > 0 else 0.0

        # Save metrics (box coords are now NaN since we do pixel-level)
        rows.append({
            'image_id': r.image_id,
            'model': r.model,
            'target_class': r.target_class,
            'subset': r.subset,
            'miou': miou,
            'pred_box_xmin': np.nan,
            'pred_box_ymin': np.nan,
            'pred_box_xmax': np.nan,
            'pred_box_ymax': np.nan
        })

    out = pd.DataFrame(rows)
    out.to_csv(RESULTS_PATH / f'miou_results_{consensus}.csv', index=False)
    print(f'✅ miou_results_{consensus}.csv saved (pixel-level IoU): {len(out)} rows from {bbox_src.name}')

# Run both — primary then sensitivity
compute_miou_file('2of3')
compute_miou_file('3of3')

import shutil
shutil.copy(RESULTS_PATH / 'miou_results_2of3.csv', RESULTS_PATH / 'miou_results.csv')
print('✅ Copied miou_results_2of3.csv to miou_results.csv for downstream steps')


✅ miou_results_2of3.csv saved (pixel-level IoU): 1008 rows from consensus_boxes_2of3.csv
✅ miou_results_3of3.csv saved (pixel-level IoU): 806 rows from consensus_boxes_3of3.csv
✅ Copied miou_results_2of3.csv to miou_results.csv for downstream steps


## Cell 4 — Load files

Load all CSV outputs used downstream.

In [ ]:
miou_path = RESULTS_PATH / 'miou_results.csv'
mono_path = RESULTS_PATH / 'monotonicity_results.csv'
lime_path = RESULTS_PATH / 'lime_ig_agreement.csv'
class_path = RESULTS_PATH / 'classification_baseline.csv'
sens_path = RESULTS_PATH / 'retrospective_sensitivity.csv'
delins_path = RESULTS_PATH / 'deletion_insertion.csv'
fleiss_path = DATA_PATH / 'fleiss_kappa.csv'

required = [miou_path, mono_path, lime_path, class_path, sens_path, delins_path, fleiss_path]
for p in required:
    assert p.exists(), f'Missing required file: {p}'

miou_df = pd.read_csv(miou_path)
mono_df = pd.read_csv(mono_path)
lime_df = pd.read_csv(lime_path)
class_df = pd.read_csv(class_path)
sens_df = pd.read_csv(sens_path)
delins_df = pd.read_csv(delins_path)
fleiss_df = pd.read_csv(fleiss_path)

print('✅ Required files loaded.')

✅ Required files loaded.


## Cell 5 — Schema checks

Verify column names before aggregation.

In [ ]:
def assert_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f'{name} missing columns: {missing}. Found: {df.columns.tolist()}'

assert_cols(miou_df, ['image_id', 'model', 'target_class', 'subset', 'miou'], 'miou_results.csv')
assert_cols(mono_df, ['image_id', 'model', 'target_class', 'monotonicity_auc'], 'monotonicity_results.csv')
assert_cols(lime_df, ['image_id', 'model', 'target_class', 'lime_ig_iou'], 'lime_ig_agreement.csv')
assert_cols(delins_df, ['image_id', 'model', 'target_class', 'deletion_auc'], 'deletion_insertion.csv')
assert_cols(fleiss_df, ['class_name', 'fleiss_kappa'], 'fleiss_kappa.csv')

if 'hit_rate' not in miou_df.columns:
    warnings.warn('hit_rate not found in miou_results.csv; it will be derived as (miou > 0).')

print('✅ Schemas validated.')

✅ Schemas validated.


/tmp/ipykernel_3214/3958487942.py:12: UserWarning: hit_rate not found in miou_results.csv; it will be derived as (miou > 0).
  warnings.warn('hit_rate not found in miou_results.csv; it will be derived as (miou > 0).')


## Cell 6 — Helpers

Bootstrap, stable seeding, and safe rounding utilities.

In [ ]:
def bootstrap_ci(values, n_boot=BOOTSTRAP_N, ci=0.95, seed=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(vals)
    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = vals[idx].mean()
    lo_q = (1 - ci) / 2
    hi_q = 1 - lo_q
    return float(boots.mean()), float(np.quantile(boots, lo_q)), float(np.quantile(boots, hi_q))

def safe_round(x, nd=4):
    return float(np.round(x, nd)) if pd.notna(x) else np.nan

def stable_seed(text):
    return int(hashlib.md5(text.encode('utf-8')).hexdigest(), 16) % 100000

## Cell 7 — Merge metrics

Combine faithfulness outputs per image.

In [ ]:
miou_use = miou_df.copy()
if 'hit_rate' not in miou_use.columns:
    miou_use['hit_rate'] = (miou_use['miou'] > 0.0).astype(float)

key_cols = ['image_id', 'model', 'target_class']

merged_df = (
    miou_use[key_cols + ['subset', 'miou', 'hit_rate']]
    .merge(mono_df[key_cols + ['monotonicity_auc']], on=key_cols, how='inner')
    .merge(lime_df[key_cols + ['lime_ig_iou']], on=key_cols, how='inner')
    .merge(delins_df[key_cols + ['deletion_auc']], on=key_cols, how='inner')
)

print('✅ merged_df shape:', merged_df.shape)

✅ merged_df shape: (1008, 9)


## Cell 8 — Mapping tables

Build lookup dictionaries for Fleiss κ and exploratory flags.

In [ ]:
kappa_map = dict(zip(fleiss_df['class_name'], fleiss_df['fleiss_kappa']))
pathology_counts = merged_df.groupby('target_class')['image_id'].nunique().to_dict()

# ── NB06 Step 5.3: Exploratory Flag Criterion ────────────────────────────
# Assert new schema — fail loudly if old power column is accidentally present
assert 'power' not in sens_df.columns, "retrospective_sensitivity.csv still has power column — regenerate from NB03"
assert 'mde_threshold' in sens_df.columns, "retrospective_sensitivity.csv missing mde_threshold column — regenerate from NB03"

MDE_THRESHOLD = sens_df['mde_threshold'].iloc[0]  # 0.05
n_test_map = dict(zip(sens_df['pathology'], sens_df['n_test']))

def flag_by_n(pathology, min_n=MIN_TEST_N):
    return n_test_map.get(pathology, 0) < min_n

print('✅ κ, MDE, and counts ready.')


✅ κ, MDE, and counts ready.


## Cell 9 — Aggregate per model × pathology

Bootstrap all primary faithfulness metrics.

In [ ]:
rows = []

for model in MODEL_ORDER:
    for patho in sorted(merged_df['target_class'].unique()):
        sub = merged_df[(merged_df['model'] == model) & (merged_df['target_class'] == patho)]
        if sub.empty:
            continue
        seed_base = SEED + stable_seed(f'{model}::{patho}')
        miou_mean, miou_lo, miou_hi = bootstrap_ci(sub['miou'].values, seed=seed_base + 1)
        hit_mean, hit_lo, hit_hi = bootstrap_ci(sub['hit_rate'].values, seed=seed_base + 2)
        mono_mean, mono_lo, mono_hi = bootstrap_ci(sub['monotonicity_auc'].values, seed=seed_base + 3)
        lime_mean, lime_lo, lime_hi = bootstrap_ci(sub['lime_ig_iou'].values, seed=seed_base + 4)
        del_mean, del_lo, del_hi = bootstrap_ci(sub['deletion_auc'].values, seed=seed_base + 5)
        n = int(sub['image_id'].nunique())
        rows.append({'model': model, 'pathology': patho, 'n': n, 'kappa': safe_round(kappa_map.get(patho, np.nan)), 'flag_proxy': '', 'miou_mean': safe_round(miou_mean), 'miou_ci_lo': safe_round(miou_lo), 'miou_ci_hi': safe_round(miou_hi), 'hit_rate_mean': safe_round(hit_mean), 'hit_rate_ci_lo': safe_round(hit_lo), 'hit_rate_ci_hi': safe_round(hit_hi), 'mono_mean': safe_round(mono_mean), 'mono_ci_lo': safe_round(mono_lo), 'mono_ci_hi': safe_round(mono_hi), 'lime_mean': safe_round(lime_mean), 'lime_ci_lo': safe_round(lime_lo), 'lime_ci_hi': safe_round(lime_hi), 'delins_mean': safe_round(del_mean), 'delins_ci_lo': safe_round(del_lo), 'delins_ci_hi': safe_round(del_hi)})

agg_df = pd.DataFrame(rows)
print('✅ agg_df shape:', agg_df.shape)

✅ agg_df shape: (26, 20)


## Cell 10 — Wilcoxon tests

Paired signed-rank tests across models.

In [ ]:
def paired_tests_for_metric(df_long, metric_name):
    out = []
    for patho in sorted(df_long['target_class'].unique()):
        sub = df_long[df_long['target_class'] == patho]
        piv = sub.pivot(index='image_id', columns='model', values=metric_name)
        piv = piv[[m for m in MODEL_ORDER if m in piv.columns]]
        if piv.shape[1] < 2:
            continue
        for i in range(len(piv.columns)):
            for j in range(i + 1, len(piv.columns)):
                m1, m2 = piv.columns[i], piv.columns[j]
                pair_df = piv[[m1, m2]].dropna()
                if len(pair_df) < WILCOX_MIN_N:
                    continue

                diff = pair_df[m1] - pair_df[m2]
                diff = diff.dropna()
                if len(diff) < WILCOX_MIN_N:
                    continue
                if np.allclose(diff.values, 0):
                    continue
                if np.unique(diff.values).size < 2:
                    continue

                try:
                    stat, p = wilcoxon(
                        pair_df[m1], pair_df[m2],
                        zero_method='wilcox',
                        alternative='two-sided',
                        mode='auto'
                    )
                except Exception as e:
                    warnings.warn(f'Wilcoxon skip {patho}/{m1}_vs_{m2}/{metric_name}: {e}')
                    continue

                out.append({
                    'pathology': patho,
                    'metric': metric_name,
                    'pair': f'{m1}_vs_{m2}',
                    'statistic': float(stat),
                    'p_raw': float(p),
                    'n_images': len(pair_df)
                })
    return out

all_tests = []
for metric in ['miou', 'monotonicity_auc', 'hit_rate', 'lime_ig_iou', 'deletion_auc']:
    all_tests.extend(paired_tests_for_metric(merged_df, metric))

wilcoxon_faith_df = pd.DataFrame(all_tests)
if not wilcoxon_faith_df.empty:
    reject, p_bonf, _, _ = multipletests(wilcoxon_faith_df['p_raw'], alpha=ALPHA, method='bonferroni')
    wilcoxon_faith_df['p_bonf'] = p_bonf
    wilcoxon_faith_df['reject'] = reject
else:
    wilcoxon_faith_df['p_bonf'] = []
    wilcoxon_faith_df['reject'] = []

print('✅ faithfulness Wilcoxon done:', wilcoxon_faith_df.shape)

✅ faithfulness Wilcoxon done: (48, 8)


## Cell 11 — Spearman confound analysis

Make the classification schema robust to either `class` or `target_class`.

In [ ]:
# ── Cell 11 — Spearman confound analysis (Image-Level) ─────────────
# Loads per-image probabilities and mIoU to compute image-level Spearman correlation.

# Load probabilities for each model
prob_dfs = {}
for model_name in MODEL_ORDER:
    fp = RESULTS_PATH / f'{model_name}_test_image_probs.csv'
    assert fp.exists(), f'Missing {fp}. Run NB03 for {model_name} first.'
    df = pd.read_csv(fp)
    # Melt it so we have (image_id, target_class, prob)
    df_long = df.melt(id_vars='image_id', var_name='target_class', value_name='prob')
    df_long['model'] = model_name
    prob_dfs[model_name] = df_long

probs_all = pd.concat(prob_dfs.values(), ignore_index=True)

# Merge with mIoU (merged_df already contains mIoU and subset=='patho')
confound_df = pd.merge(merged_df, probs_all, on=['image_id', 'model', 'target_class'], how='inner')

spearman_rows = []

# 1. Per-model correlation (absolute)
print("── Absolute Image-Level Spearman (prob vs mIoU) ──")
for m in MODEL_ORDER:
    sub = confound_df[confound_df['model'] == m]
    valid = sub['prob'].notna() & sub['miou'].notna()
    if valid.sum() < 10: continue
    rho, p = spearmanr(sub.loc[valid, 'prob'], sub.loc[valid, 'miou'])
    print(f"{m:16s} : rho={rho:+.3f}, p={p:.3f} (n={valid.sum()})")
    spearman_rows.append({'type': 'absolute', 'model_or_pair': m, 'n': int(valid.sum()), 'rho': float(rho), 'p_value': float(p)})

# 2. Paired correlation (Δprob vs ΔmIoU)
print("\n── Paired Image-Level Spearman (Δprob vs ΔmIoU) ──")
for i in range(len(MODEL_ORDER)):
    for j in range(i + 1, len(MODEL_ORDER)):
        m1, m2 = MODEL_ORDER[i], MODEL_ORDER[j]
        sub1 = confound_df[confound_df['model'] == m1].set_index(['image_id', 'target_class'])
        sub2 = confound_df[confound_df['model'] == m2].set_index(['image_id', 'target_class'])

        common = sub1.index.intersection(sub2.index)
        if len(common) < 10: continue

        d_prob = sub1.loc[common, 'prob'] - sub2.loc[common, 'prob']
        d_miou = sub1.loc[common, 'miou'] - sub2.loc[common, 'miou']

        valid = d_prob.notna() & d_miou.notna()
        rho, p = spearmanr(d_prob[valid], d_miou[valid])
        print(f"{m1} vs {m2} : rho={rho:+.3f}, p={p:.3f} (n={valid.sum()})")
        spearman_rows.append({'type': 'paired_delta', 'model_or_pair': f'{m1}_vs_{m2}', 'n': int(valid.sum()), 'rho': float(rho), 'p_value': float(p)})

spearman_df = pd.DataFrame(spearman_rows)

# ── NB06 Step 5.4: CI-Based Divergence Criterion ─────────────────────────

# Replaces previous power reclassification entirely.
# A pathology-pair is DIVERGENT iff:
#   (1) 95% bootstrap CI on mean Δ(mIoU) excludes zero, AND
#   (2) |point estimate| >= MDE_THRESHOLD (0.05)

def bootstrap_ci_diff(diffs, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    boot_means = [np.mean(rng.choice(diffs, size=len(diffs), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_means, 100 * alpha / 2)
    hi = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return lo, hi

divergence_records = []
exploratory_ci_pathos = set()

for patho in sorted(merged_df['target_class'].unique()):
    for i in range(len(MODEL_ORDER)):
        for j in range(i + 1, len(MODEL_ORDER)):
            m1, m2 = MODEL_ORDER[i], MODEL_ORDER[j]
            sub1 = merged_df[(merged_df['model'] == m1) & (merged_df['target_class'] == patho)]
            sub2 = merged_df[(merged_df['model'] == m2) & (merged_df['target_class'] == patho)]

            # Align by image_id
            df12 = sub1[['image_id', 'miou']].merge(sub2[['image_id', 'miou']], on='image_id', suffixes=('_1', '_2'))
            if len(df12) < MIN_TEST_N:
                exploratory_ci_pathos.add(patho)
                continue

            diffs = df12['miou_1'].values - df12['miou_2'].values
            point_est = np.mean(diffs)
            ci_lo, ci_hi = bootstrap_ci_diff(diffs)

            ci_excludes_0 = (ci_lo > 0) or (ci_hi < 0)
            exceeds_mde = abs(point_est) >= MDE_THRESHOLD
            is_divergent = ci_excludes_0 and exceeds_mde

            if not ci_excludes_0:
                exploratory_ci_pathos.add(patho)

            divergence_records.append({
                'pathology': patho,
                'model_a': m1,
                'model_b': m2,
                'point_estimate': round(point_est, 4),
                'ci_low': round(ci_lo, 4),
                'ci_high': round(ci_hi, 4),
                'ci_excludes_zero': ci_excludes_0,
                'exceeds_mde': exceeds_mde,
                'is_divergent': is_divergent
            })

div_df = pd.DataFrame(divergence_records)
div_df.to_csv(RESULTS_PATH / 'divergence_analysis.csv', index=False)

print("✅ Divergence summary generated.")
print(div_df[['pathology','model_a','model_b','point_estimate','ci_low','ci_high','is_divergent']].head().to_string(index=False))


── Absolute Image-Level Spearman (prob vs mIoU) ──
densenet121      : rho=+0.145, p=0.009 (n=324)
convnextv2_tiny  : rho=+0.359, p=0.000 (n=333)
swinb_lora       : rho=+0.116, p=0.029 (n=351)

── Paired Image-Level Spearman (Δprob vs ΔmIoU) ──
densenet121 vs convnextv2_tiny : rho=+0.232, p=0.001 (n=219)
densenet121 vs swinb_lora : rho=+0.283, p=0.000 (n=239)
convnextv2_tiny vs swinb_lora : rho=+0.158, p=0.012 (n=251)
✅ Divergence summary generated.
         pathology         model_a         model_b  point_estimate  ci_low  ci_high  is_divergent
Aortic enlargement     densenet121 convnextv2_tiny          0.0058  0.0024   0.0098         False
Aortic enlargement     densenet121      swinb_lora         -0.0448 -0.0483  -0.0411         False
Aortic enlargement convnextv2_tiny      swinb_lora         -0.0507 -0.0546  -0.0468          True
      Cardiomegaly     densenet121 convnextv2_tiny          0.0079 -0.0067   0.0221         False
      Cardiomegaly convnextv2_tiny      swinb_lora       

## Cell 12 — Rare cluster pooling

Pool pathologies with fewer than `MIN_TEST_N` images.

In [ ]:
rare_pathos = [p for p, n in pathology_counts.items() if n < MIN_TEST_N]
rare_rows = []

if rare_pathos:
    for model in MODEL_ORDER:
        sub = merged_df[(merged_df['model'] == model) & (merged_df['target_class'].isin(rare_pathos))]
        if sub.empty:
            continue
        seed_base = SEED + stable_seed(f'{model}::rare_cluster')
        miou_mean, miou_lo, miou_hi = bootstrap_ci(sub['miou'].values, seed=seed_base + 1)
        hit_mean, hit_lo, hit_hi = bootstrap_ci(sub['hit_rate'].values, seed=seed_base + 2)
        mono_mean, mono_lo, mono_hi = bootstrap_ci(sub['monotonicity_auc'].values, seed=seed_base + 3)
        lime_mean, lime_lo, lime_hi = bootstrap_ci(sub['lime_ig_iou'].values, seed=seed_base + 4)
        del_mean, del_lo, del_hi = bootstrap_ci(sub['deletion_auc'].values, seed=seed_base + 5)
        n = int(sub['image_id'].nunique())
        kappas = [kappa_map.get(p, np.nan) for p in rare_pathos]
        rare_rows.append({'model': model, 'pathology': 'rare_cluster', 'n': n, 'kappa': safe_round(np.nanmean(kappas)), 'flag_proxy': '', 'miou_mean': safe_round(miou_mean), 'miou_ci_lo': safe_round(miou_lo), 'miou_ci_hi': safe_round(miou_hi), 'hit_rate_mean': safe_round(hit_mean), 'hit_rate_ci_lo': safe_round(hit_lo), 'hit_rate_ci_hi': safe_round(hit_hi), 'mono_mean': safe_round(mono_mean), 'mono_ci_lo': safe_round(mono_lo), 'mono_ci_hi': safe_round(mono_hi), 'lime_mean': safe_round(lime_mean), 'lime_ci_lo': safe_round(lime_lo), 'lime_ci_hi': safe_round(lime_hi), 'delins_mean': safe_round(del_mean), 'delins_ci_lo': safe_round(del_lo), 'delins_ci_hi': safe_round(del_hi)})

rare_df = pd.DataFrame(rare_rows)
print('✅ rare cluster done.')

✅ rare cluster done.


## Cell 13 — Final tables

Assemble export-ready summary tables.

In [ ]:
table_df = agg_df.copy()
if not rare_df.empty:
    table_df = pd.concat([table_df, rare_df], ignore_index=True, sort=False)

# Final exploratory flag: criterion A (n < 30) OR criterion B (CI includes zero)
def get_final_flag(row):
    is_low_n = pd.notna(row['n']) and row['n'] < MIN_TEST_N
    is_ci_fail = row['pathology'] in exploratory_ci_pathos
    if is_low_n or is_ci_fail:
        return '†'
    return ''

table_df['flag'] = table_df.apply(get_final_flag, axis=1)

# Backwards compatibility for existing print statements or plots
table_df['n_flag'] = table_df['flag']
table_df['flag_proxy'] = table_df['flag']

summary_rows = []
for (patho, metric), g in wilcoxon_faith_df.groupby(['pathology', 'metric']):
    summary_rows.append({'pathology': patho, 'metric': metric, 'n_tests': int(len(g)), 'p_bonf_min': float(g['p_bonf'].min()), 'pairs': '; '.join(f'{r.pair}:{r.p_bonf:.4g}' for r in g.itertuples())})

wilcoxon_summary_df = pd.DataFrame(summary_rows)
print('✅ Final tables assembled.')


✅ Final tables assembled.


## Cell 14 — Save outputs

Write the CSV files used by the manuscript and later table-building steps.

In [ ]:
faith_path = RESULTS_PATH / 'faithfulness_results.csv'
spearman_path = RESULTS_PATH / 'spearman_confounds.csv'
wilcox_path = RESULTS_PATH / 'wilcoxon_faithfulness.csv'
wilcox_summ_path = RESULTS_PATH / 'wilcoxon_faithfulness_summary.csv'

table_df.to_csv(faith_path, index=False)
spearman_df.to_csv(spearman_path, index=False)
wilcoxon_faith_df.to_csv(wilcox_path, index=False)
wilcoxon_summary_df.to_csv(wilcox_summ_path, index=False)

print('✅ All outputs saved.')

✅ All outputs saved.


## Cell 15 — Sanity checks

Compact verification summaries before manuscript export.

In [ ]:
print('Mean mIoU by model:')
print(table_df.groupby('model')['miou_mean'].mean().round(4).to_string())

print('\nWilcoxon rows:')
print(wilcoxon_faith_df.head().to_string(index=False))

print('\nSpearman rows:')
print(spearman_df.to_string(index=False))

Mean mIoU by model:
model
convnextv2_tiny    0.1210
densenet121        0.0890
swinb_lora         0.1629

Wilcoxon rows:
         pathology metric                           pair  statistic        p_raw  n_images       p_bonf  reject
Aortic enlargement   miou densenet121_vs_convnextv2_tiny     3423.0 1.037599e-02       135 4.980475e-01   False
Aortic enlargement   miou      densenet121_vs_swinb_lora        9.0 2.828996e-27       156 1.357918e-25    True
Aortic enlargement   miou  convnextv2_tiny_vs_swinb_lora        0.0 2.299606e-26       150 1.103811e-24    True
      Cardiomegaly   miou densenet121_vs_convnextv2_tiny      243.0 3.604973e-01        34 1.000000e+00   False
      Cardiomegaly   miou      densenet121_vs_swinb_lora       13.0 3.278255e-07        29 1.573563e-05    True

Spearman rows:
        type                  model_or_pair   n      rho      p_value
    absolute                    densenet121 324 0.144640 9.129094e-03
    absolute                convnextv2_tiny 333 0.35

In [ ]:
# Sensitivity analysis: mIoU stability across consensus thresholds
sens_rows = []
for consensus in ['2of3', '3of3']:
    p = RESULTS_PATH / f'miou_results_{consensus}.csv'
    if not p.exists():
        warnings.warn(f'Missing {p.name}, skipping.')
        continue
    df = pd.read_csv(p)
    df['consensus'] = consensus
    agg = df.groupby(['model', 'target_class', 'consensus'])['miou'].agg(
        miou_mean='mean', miou_std='std', n='count'
    ).reset_index()
    sens_rows.append(agg)

if sens_rows:
    sens_df = pd.concat(sens_rows, ignore_index=True)
    sens_df.to_csv(RESULTS_PATH / 'sensitivity_analysis.csv', index=False)
    print('✅ sensitivity_analysis.csv saved:', sens_df.shape)

    # Quick consistency check
    pivot = sens_df.pivot_table(
        index=['model','target_class'], columns='consensus', values='miou_mean'
    )
    pivot['delta'] = pivot['3of3'] - pivot['2of3']
    print('\nmIoU delta (3of3 minus 2of3) by model:')
    print(pivot.groupby('model')['delta'].mean().round(4))

✅ sensitivity_analysis.csv saved: (50, 6)

mIoU delta (3of3 minus 2of3) by model:
model
convnextv2_tiny    0.0106
densenet121        0.0030
swinb_lora         0.0097
Name: delta, dtype: float64


##"mIoU results were robust to annotation consensus threshold. Mean mIoU differed by less than 0.013 across all models when comparing 2-of-3 and 3-of-3 radiologist agreement criteria (Table A2), indicating findings are not driven by annotation noise."

In [ ]:
sens_df = pd.read_csv(RESULTS_PATH / 'sensitivity_analysis.csv')
by_consensus = sens_df.groupby('consensus')['target_class'].nunique()
print(by_consensus)

missing_in_3of3 = (
    set(sens_df[sens_df['consensus']=='2of3']['target_class']) -
    set(sens_df[sens_df['consensus']=='3of3']['target_class'])
)
print('Pathologies in 2of3 but dropped in 3of3:', missing_in_3of3)

consensus
2of3    10
3of3    10
Name: target_class, dtype: int64
Pathologies in 2of3 but dropped in 3of3: set()


##Nodule/Mass was excluded from the 3-of-3 sensitivity analysis as no cases met the unanimous annotator agreement criterion, consistent with known inter-reader variability for this finding

In [7]:
# ── NB06 add-on: mIoU threshold sweep (top-30% / top-50% / top-70% IG mass) ──
# Self-sufficient diagnostic — does NOT overwrite miou_results.csv

from pathlib import Path
import numpy as np
import pandas as pd

# ── Paths (Colab / NB06 convention) ─────────────────────────────────────────
try:
    ROOT         # already defined in NB06
    RESULTS_PATH
    ANN_PATH
except NameError:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_ROOT  = '/content/drive/MyDrive/cxr_faithfulness'
    exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())
    ROOT         = Path(GDRIVE_ROOT)
    RESULTS_PATH = ROOT / 'results'
    ANN_PATH     = ROOT / 'data' / 'processed' / 'consensus'

MASK_RES     = 224
CONSENSUS    = '2of3'   # change to '3of3' for sensitivity
MODEL_ORDER  = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
MASS_THRESHS = [
    ('top30', 'top30_path', 0.30),
    ('top50', 'top50_path', 0.50),
    ('top70', 'top70_path', 0.70),
]

# ── Helpers (same logic as NB06 mIoU cell) ───────────────────────────────────
def load_bbox_table(consensus=CONSENSUS):
    path = ANN_PATH / f'consensus_boxes_{consensus}.csv'
    assert path.exists(), f'Missing: {path}'
    return pd.read_csv(path), path

def find_box_columns(df):
    colmap = {}
    aliases = {
        'image_id':     ['image_id', 'id'],
        'target_class': ['target_class', 'class', 'pathology', 'class_name'],
        'xmin':         ['xmin', 'x1', 'left', 'x_min'],
        'ymin':         ['ymin', 'y1', 'top', 'y_min'],
        'xmax':         ['xmax', 'x2', 'right', 'x_max'],
        'ymax':         ['ymax', 'y2', 'bottom', 'y_max'],
    }
    for target, opts in aliases.items():
        found = next((c for c in opts if c in df.columns), None)
        if found is None:
            raise ValueError(f'Missing bbox column for {target}. Found: {df.columns.tolist()}')
        colmap[target] = found
    return colmap

def load_pred_mask(mask_path: Path) -> np.ndarray:
    arr = np.load(str(mask_path))
    if arr.ndim == 3:
        arr = arr.squeeze()
    assert arr.shape == (MASK_RES, MASK_RES), f'Bad mask shape {arr.shape} @ {mask_path.name}'
    unique = np.unique(arr)
    assert set(unique.tolist()).issubset({0, 1}), (
        f'Expected binary 0/1 mask, got {unique} @ {mask_path.name}'
    )
    return arr.astype(np.uint8)

def rasterize_gt_boxes(gt_df, cols) -> np.ndarray:
    gt_mask = np.zeros((MASK_RES, MASK_RES), dtype=np.uint8)
    for _, g in gt_df.iterrows():
        x1 = int(max(0, float(g[cols['xmin']])))
        y1 = int(max(0, float(g[cols['ymin']])))
        x2 = int(min(MASK_RES, float(g[cols['xmax']])))
        y2 = int(min(MASK_RES, float(g[cols['ymax']])))
        if x1 < x2 and y1 < y2:
            gt_mask[y1:y2, x1:x2] = 1
    return gt_mask

def pixel_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    inter = np.logical_and(pred_mask, gt_mask).sum()
    uni   = np.logical_or(pred_mask, gt_mask).sum()
    return 0.0 if uni == 0 else float(inter / uni)

# ── Main sweep ───────────────────────────────────────────────────────────────
manifest_path = RESULTS_PATH / 'ig_manifest.csv'
assert manifest_path.exists(), 'Run NB04 first — ig_manifest.csv missing'

manifest = pd.read_csv(manifest_path)
bbox_df, bbox_src = load_bbox_table(CONSENSUS)
cols = find_box_columns(bbox_df)

rows = []
skipped = {'no_box': 0, 'no_mask': 0, 'not_patho': 0}

for r in manifest.itertuples(index=False):
    if r.subset != 'patho':
        skipped['not_patho'] += 1
        continue

    gt = bbox_df[
        (bbox_df[cols['image_id']] == r.image_id) &
        (bbox_df[cols['target_class']] == r.target_class)
    ]
    if gt.empty:
        skipped['no_box'] += 1
        continue

    gt_mask = rasterize_gt_boxes(gt, cols)
    gt_area = int(gt_mask.sum())

    for mass_label, path_col, mass_frac in MASS_THRESHS:
        mask_path = Path(getattr(r, path_col))
        if not mask_path.exists():
            skipped['no_mask'] += 1
            continue

        pred_mask  = load_pred_mask(mask_path)
        pred_area  = int(pred_mask.sum())
        iou        = pixel_iou(pred_mask, gt_mask)

        rows.append({
            'image_id'     : r.image_id,
            'model'        : r.model,
            'target_class' : r.target_class,
            'mass_label'   : mass_label,
            'mass_fraction': mass_frac,
            'miou'         : round(iou, 6),
            'pred_area_px' : pred_area,
            'gt_area_px'   : gt_area,
            'area_ratio'   : round(pred_area / gt_area, 4) if gt_area > 0 else np.nan,
        })

sweep_df = pd.DataFrame(rows)
assert not sweep_df.empty, 'No rows computed — check Drive paths / mask files'

out_long  = RESULTS_PATH / f'miou_threshold_sweep_{CONSENSUS}.csv'
out_model = RESULTS_PATH / f'miou_threshold_sweep_by_model_{CONSENSUS}.csv'
out_patho = RESULTS_PATH / f'miou_threshold_sweep_by_pathology_{CONSENSUS}.csv'

sweep_df.to_csv(out_long, index=False)

# ── Summaries ────────────────────────────────────────────────────────────────
model_summary = (
    sweep_df.groupby(['mass_label', 'model'], as_index=False)
    .agg(
        n=('miou', 'count'),
        miou_mean=('miou', 'mean'),
        miou_std=('miou', 'std'),
        pred_area_mean=('pred_area_px', 'mean'),
        area_ratio_mean=('area_ratio', 'mean'),
    )
    .sort_values(['mass_label', 'model'])
)
model_summary.to_csv(out_model, index=False)

patho_summary = (
    sweep_df.groupby(['mass_label', 'model', 'target_class'], as_index=False)
    .agg(n=('miou', 'count'), miou_mean=('miou', 'mean'))
    .sort_values(['mass_label', 'model', 'target_class'])
)
patho_summary.to_csv(out_patho, index=False)

# Rank models within each threshold (1 = highest mIoU)
rank_df = model_summary.copy()
rank_df['rank'] = rank_df.groupby('mass_label')['miou_mean'].rank(ascending=False, method='min')

print(f'✅ Sweep complete — consensus: {bbox_src.name}')
print(f'   Rows (long): {len(sweep_df)}')
print(f'   Skipped — not patho: {skipped["not_patho"]}, no box: {skipped["no_box"]}, missing mask: {skipped["no_mask"]}')
print(f'   Saved: {out_long.name}')
print(f'          {out_model.name}')
print(f'          {out_patho.name}')

print('\n── Mean mIoU by model × IG mass threshold ──')
pivot = model_summary.pivot(index='model', columns='mass_label', values='miou_mean')
pivot = pivot.reindex(columns=[m[0] for m in MASS_THRESHS], index=MODEL_ORDER)
print(pivot.round(4).to_string())

print('\n── Model rank per threshold (1 = best) ──')
print(rank_df.pivot(index='model', columns='mass_label', values='rank')
      .reindex(index=MODEL_ORDER, columns=[m[0] for m in MASS_THRESHS])
      .to_string())

print('\n── Aortic enlargement only (largest reliable stratum) ──')
ae = patho_summary[patho_summary['target_class'] == 'Aortic enlargement']
if not ae.empty:
    print(ae.pivot(index='model', columns='mass_label', values='miou_mean')
          .reindex(index=MODEL_ORDER, columns=[m[0] for m in MASS_THRESHS])
          .round(4).to_string())
else:
    print('   (no rows)')

print('\n── Mean pred_area / gt_area (geometry confound check) ──')
geom = model_summary.pivot(index='model', columns='mass_label', values='area_ratio_mean')
print(geom.reindex(index=MODEL_ORDER, columns=[m[0] for m in MASS_THRESHS]).round(2).to_string())
print('   >> area_ratio >> 1 means IG mask is much larger than the radiologist box')

# Quick read: did Swin stay #1 at all thresholds?
ranks = rank_df.pivot(index='model', columns='mass_label', values='rank').reindex(MODEL_ORDER)
swin_top1 = (ranks.loc['swinb_lora'] == 1).all()
print(f'\n── Stability check: Swin rank #1 at all thresholds? {swin_top1} ──')

✅ Sweep complete — consensus: consensus_boxes_2of3.csv
   Rows (long): 3024
   Skipped — not patho: 145, no box: 375, missing mask: 0
   Saved: miou_threshold_sweep_2of3.csv
          miou_threshold_sweep_by_model_2of3.csv
          miou_threshold_sweep_by_pathology_2of3.csv

── Mean mIoU by model × IG mass threshold ──
mass_label        top30   top50   top70
model                                  
densenet121      0.0518  0.0743  0.0846
convnextv2_tiny  0.0632  0.0844  0.0924
swinb_lora       0.1341  0.1272  0.1114

── Model rank per threshold (1 = best) ──
mass_label       top30  top50  top70
model                               
densenet121        3.0    3.0    3.0
convnextv2_tiny    2.0    2.0    2.0
swinb_lora         1.0    1.0    1.0

── Aortic enlargement only (largest reliable stratum) ──
mass_label        top30   top50   top70
model                                  
densenet121      0.0294  0.0373  0.0374
convnextv2_tiny  0.0224  0.0269  0.0283
swinb_lora       0.1179  0.0798 